<a href="https://colab.research.google.com/github/keivernunez/dataminingavanzado_austral/blob/main/Clase2_Note1_de_4_Kernel_Trick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# El Truco del Kernel (Kernel Trick) en SVM

**Introducción:**

Esta sección se dedica a uno de los conceptos más potentes y elegantes de las Máquinas de Vectores de Soporte: el **truco del kernel**. Se abordará el problema de los datos que no son linealmente separables y se mostrará, a través de gráficos estáticos y una animación, cómo las SVM resuelven este desafío de manera computacionalmente eficiente.

El objetivo es proporcionar una comprensión intuitiva y visual para que los estudiantes puedan internalizar:
1.  Por qué los clasificadores lineales a veces no son suficientes.
2.  La idea de transformar datos a un espacio de mayor dimensionalidad.
3.  Cómo el truco del kernel permite lograr esta separación sin el costo computacional de realizar la transformación explícitamente.

In [ ]:
# Importaciones necesarias para visualización y modelado
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from IPython.display import HTML

# Configuraciones generales
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## El Problema: Datos No Linealmente Separables

No todos los conjuntos de datos pueden ser separados por una simple línea recta (o un hiperplano). Consideremos el siguiente ejemplo, donde una clase de puntos se encuentra rodeada por otra. Es evidente que ningún clasificador lineal podrá trazar una frontera de decisión adecuada.

In [ ]:
# 1. Generación de datos no separables linealmente
X, y = make_circles(n_samples=200, factor=0.3, noise=0.1, random_state=42)

# 2. Entrenar un clasificador SVM lineal
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X, y)

# 3. Función para visualizar la frontera de decisión (Gráfico 1)
def plot_decision_boundary(clf, X, y, title):
    plt.figure(figsize=(8, 8))
    plt.scatter(X[:, 0], X[:, 1], c=y, s=50, cmap='viridis', edgecolors='k')
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # Crear una grilla para evaluar el modelo
    xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 50),
                         np.linspace(ylim[0], ylim[1], 50))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Dibujar la frontera de decisión y los márgenes
    ax.contour(xx, yy, Z, colors='k', levels=[-1, 0, 1], alpha=0.5,
               linestyles=['--', '-', '--'])

    # Resaltar los vectores de soporte
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100,
               linewidth=1, facecolors='none', edgecolors='k')
    plt.title(title, fontsize=16)
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.show()

# Gráfico 1: El fracaso del clasificador lineal
plot_decision_boundary(svm_linear, X, y, 'Frontera de Decisión de un SVM Lineal (Resultado Incorrecto)')

## La Solución: Proyectar a una Dimensión Superior

La idea clave es transformar los datos a un espacio de mayor dimensionalidad donde sí sean linealmente separables. Para nuestros datos en 2D (coordenadas $x_1, x_2$), podemos definir una tercera dimensión, $z$, que sea una función de las dimensiones originales. Una transformación común es la **función de base radial (RBF)**, que se puede visualizar de forma simplificada añadiendo una tercera dimensión $z = x_1^2 + x_2^2$.

Al aplicar esta transformación, los puntos del círculo interior (cercanos al origen) tendrán un valor de $z$ bajo, mientras que los puntos del círculo exterior tendrán un valor de $z$ alto. En este nuevo espacio 3D, los datos ahora pueden ser separados por un simple plano.

In [ ]:
# Gráfico 2: Datos proyectados a 3D
# Se añade una tercera dimensión z = x^2 + y^2
X_3d = np.hstack((X, (X[:, 0]**2 + X[:, 1]**2)[:, np.newaxis]))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=y, s=50, cmap='viridis', edgecolors='k')
ax.set_title('Datos Proyectados en un Espacio 3D', fontsize=16)
ax.set_xlabel('Característica 1')
ax.set_ylabel('Característica 2')
ax.set_zlabel('Característica 3 (x1^2 + x2^2)')

# Se crea un plano de separación artificial para ilustrar el concepto
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 50), np.linspace(-1.5, 1.5, 50))
zz = np.ones_like(xx) * 0.5 # Plano en z=0.5
ax.plot_surface(xx, yy, zz, alpha=0.3, color='gray', rstride=10, cstride=10)

ax.view_init(elev=20, azim=20)
plt.show()

### El "Truco" del Kernel

Calcular explícitamente estas transformaciones para cada punto puede ser computacionalmente muy costoso, especialmente si se mapea a espacios de dimensiones infinitas. Aquí es donde reside la genialidad del **truco del kernel**.

Un **kernel** es una función que calcula el producto escalar (una medida de similitud) entre dos puntos en ese espacio de alta dimensionalidad, **sin tener que realizar la transformación explícita**. El algoritmo de SVM solo necesita estos productos escalares para encontrar el hiperplano óptimo. De esta forma, obtenemos el poder de trabajar en un espacio de características complejo sin el costo computacional asociado.

El kernel más popular es el **RBF (`rbf`)**, que corresponde a mapear a un espacio de infinitas dimensiones.

In [ ]:
# Gráfico 3: El resultado del Kernel Trick en el espacio original
# Se entrena un SVM con el kernel RBF en los datos 2D originales
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='auto', random_state=42)
svm_rbf.fit(X, y)

# Se visualiza la frontera de decisión no lineal que el kernel encontró
plot_decision_boundary(svm_rbf, X, y, 'Frontera de Decisión con Kernel RBF (Resultado Correcto)')

## Animación: De 2D a 3D

La siguiente animación consolida todo el proceso. Muestra los datos originales en 2D, luego los "eleva" a la tercera dimensión para hacerlos linealmente separables, y finalmente muestra el plano de separación en 3D. Esto sirve para ilustrar de manera dinámica el concepto fundamental detrás del truco del kernel.

In [ ]:
# Código para generar la animación
fig_anim = plt.figure(figsize=(10, 8))
ax_anim = fig_anim.add_subplot(111, projection='3d')

z_coords = X[:, 0]**2 + X[:, 1]**2

# Estado inicial (puntos en el plano z=0)
scatter = ax_anim.scatter(X[:, 0], X[:, 1], np.zeros_like(z_coords), c=y, s=50, cmap='viridis', edgecolors='k')
ax_anim.set_title('Animación del Mapeo de Características', fontsize=16)
ax_anim.set_xlabel('X1')
ax_anim.set_ylabel('X2')
ax_anim.set_zlabel('Z = X1^2 + X2^2')
ax_anim.set_zlim(0, 2)

# Plano de separación (inicialmente invisible)
xx_p, yy_p = np.meshgrid(np.linspace(-1.5, 1.5, 10), np.linspace(-1.5, 1.5, 10))
zz_p = np.ones_like(xx_p) * 0.5
plane = [ax_anim.plot_surface(xx_p, yy_p, zz_p, alpha=0.0, color='gray')]

def update(frame):
    # Fase 1: Elevar los puntos (frames 0-50)
    if frame <= 50:
        progress = frame / 50
        current_z = z_coords * progress
        scatter._offsets3d = (X[:, 0], X[:, 1], current_z)
        ax_anim.view_init(elev=20, azim=30)

    # Fase 2: Mostrar el plano (frame 51)
    elif frame == 51:
        plane[0].set_alpha(0.3)

    # Fase 3: Rotar la vista (frames 52-150)
    else:
        progress = (frame - 52) / (150 - 52)
        ax_anim.view_init(elev=20, azim=30 + progress * 360)

    return scatter, plane[0]

# Crear la animación
anim = FuncAnimation(fig_anim, update, frames=150, blit=False, interval=50)

# Convertir la animación a HTML para mostrarla en el notebook
plt.close(fig_anim) # Evitar que se muestre el frame estático final
HTML(anim.to_jshtml())

# Conclusión Final y Síntesis del Recorrido

A lo largo de esta serie de cuadernos, se ha realizado un viaje completo a través de las Máquinas de Vectores de Soporte, desde sus fundamentos teóricos hasta su aplicación práctica y optimización. El recorrido se puede resumir en tres etapas clave:

1.  **La Base Fundamental (Cuaderno 1):** Se establecieron los pilares conceptuales de las SVM: la búsqueda del **hiperplano de margen máximo**, la importancia de los **vectores de soporte** y el equilibrio entre sesgo y varianza a través del hiperparámetro de regularización **`C`**. Se aplicaron estos conceptos a un problema de clasificación real (el dataset del Titanic) utilizando un flujo de trabajo profesional con `Pipelines` y `GridSearchCV`.

2.  **La Optimización por Hardware (Cuaderno 2):** Se abordó un desafío práctico en el Machine Learning moderno: el tiempo de cómputo. Se demostró cómo la migración del entrenamiento de una CPU a una **GPU** mediante la librería **RAPIDS cuML** puede generar una **aceleración (speedup) significativa**, haciendo factible el trabajo con datasets de mayor tamaño. Esta etapa subrayó que la elección del hardware y las librerías es una parte crucial del proceso de modelado a escala.

3.  **La Abstracción Elegante (Cuaderno 3):** Se profundizó en el concepto más poderoso y distintivo de las SVM: el **truco del kernel**. A través de visualizaciones y una animación, se desmitificó cómo las SVM manejan datos no lineales. Se ilustró que, en lugar de realizar transformaciones complejas y costosas, el kernel permite operar en un espacio de características de alta dimensionalidad de manera implícita, encontrando fronteras de decisión complejas con una eficiencia notable.

---

## ¿Cuándo Utilizar SVM?

Las SVM son una herramienta excepcionalmente potente, pero como todo algoritmo, tienen escenarios donde brillan y otros donde pueden no ser la mejor opción.

✅ **Fortalezas y Casos de Uso Ideales:**
- **Espacios de Alta Dimensionalidad:** Son particularmente efectivas cuando la cantidad de características es grande en comparación con el número de muestras (ej. análisis de texto, bioinformática).
- **Problemas Complejos pero con Datos Limitados:** Cuando no se dispone de un volumen masivo de datos (donde las redes neuronales profundas suelen destacar), las SVM con un kernel bien ajustado (como el RBF) pueden ofrecer un rendimiento de vanguardia.
- **Eficiencia de Memoria:** El modelo final solo depende de los vectores de soporte, lo que lo hace muy eficiente en términos de memoria al momento de realizar predicciones.
- **Robustez al Sobreajuste:** La maximización del margen es un mecanismo de regularización inherente que ayuda a que el modelo generalice bien a datos no vistos.

⚠️ **Debilidades y Consideraciones:**
- **Costo Computacional:** El entrenamiento puede ser lento en datasets muy grandes (cientos de miles de muestras o más), ya que su complejidad algorítmica es sensible al tamaño del conjunto de datos. La aceleración por GPU, como se vio, es una solución clave a este problema.
- **Sensibilidad a Hiperparámetros:** La elección del kernel y la correcta sintonización de los hiperparámetros (`C`, `gamma`) son cruciales para un buen rendimiento. Un `GridSearchCV` o una búsqueda aleatoria son casi siempre necesarios.
- **Interpretabilidad:** A diferencia de los árboles de decisión, los modelos SVM, especialmente con kernels no lineales, funcionan como una "caja negra", haciendo más difícil la interpretación de sus resultados.

En resumen, las Máquinas de Vectores de Soporte representan una combinación notable de elegancia teórica y eficacia práctica. Dominar sus conceptos no solo añade una herramienta poderosa al arsenal de cualquier científico de datos, sino que también proporciona una visión profunda sobre los principios de optimización, regularización y manejo de la dimensionalidad que son fundamentales en todo el campo del Machine Learning.